In [3]:
pip install transformers datasets evaluate scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import random
import numpy as np
import evaluate
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

# 1. Imitate utils.py
def load_hf_dataset(description, split_name):
    """Load and format OOD datasets."""
    print(f"Loading {description} ({split_name})...")

    if description == 'pietrolesci/nli_fever':
        ds = load_dataset(description, split=split_name)
        ds = ds.rename_column("premise", "new_hypothesis")
        ds = ds.rename_column("hypothesis", "premise")
        ds = ds.rename_column("new_hypothesis", "hypothesis")
        return ds

    elif description == 'alisawuffles/WANLI':
        ds = load_dataset(description, split=split_name)
        labels = [0 if g=='entailment' else 1 if g=='neutral' else 2 for g in ds['gold']]
        ds = ds.add_column("label", labels)
        return ds

    elif description == 'allenai/scitail':
        ds = load_dataset(description, 'snli_format', split=split_name)
        labels = [0 if g in ['entails', 'entailment'] else 1 for g in ds['gold_label']]
        ds = ds.add_column("label", labels)
        ds = ds.rename_column("sentence1", "premise")
        ds = ds.rename_column("sentence2", "hypothesis")
        return ds

    else:
        return load_dataset(description, split=split_name)

# 2. Setup OOD Test Datasets
ood_configs = [
    ("snli", "test"),
    ("multi_nli", "validation_matched"),
    ("anli", "test_r1"),
    ("allenai/scitail", "test"),
    ("alisawuffles/WANLI", "test")
]

ood_datasets = {}
for desc, split in ood_configs:
    ds = load_hf_dataset(desc, split)
    ds = ds.filter(lambda x: x['label'] in [0, 1, 2])
    # Take 1000 samples
    if len(ds) > 1000: ds = ds.select(range(1000))
    ood_datasets[f"{desc}_{split}"] = ds

# 3. Setup Tokenizer & Metrics
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    texts = [f"Premise: {p}\nHypothesis: {h}" for p, h in zip(examples['premise'], examples['hypothesis'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

print("Tokenizing OOD sets...")
tok_ood = {k: v.map(tokenize_fn, batched=True) for k, v in ood_datasets.items()}

acc_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    return acc_metric.compute(predictions=preds, references=eval_pred.label_ids)

/Users/zhoulinrong/Desktop/COM3610 Dissertation Project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading snli (test)...
Loading multi_nli (validation_matched)...
Loading anli (test_r1)...
Loading allenai/scitail (test)...


Loading alisawuffles/WANLI (test)...
Tokenizing OOD sets...


Map: 100%|██████████| 1000/1000 [00:00<00:00, 24118.50 examples/s]


In [5]:
# 4. Process Multiple Random 10k Splits
random_seeds = [13, 42, 100]
all_results = []

# Base training data
print("Loading base SNLI train data...")
snli_train_full = load_dataset("snli", split="train").filter(lambda x: x['label'] in [0, 1, 2])

for run_idx, seed in enumerate(random_seeds):
    print(f"\n=Starting Run {run_idx + 1} / {len(random_seeds)} (Seed: {seed}) ===")

    set_seed(seed)
    random.seed(seed)

    # Sample 10k balanced
    print("Sampling 10k balanced...")
    c0 = snli_train_full.filter(lambda x: x['label'] == 0).shuffle(seed=seed).select(range(3333))
    c1 = snli_train_full.filter(lambda x: x['label'] == 1).shuffle(seed=seed).select(range(3333))
    c2 = snli_train_full.filter(lambda x: x['label'] == 2).shuffle(seed=seed).select(range(3334))
    train_10k = concatenate_datasets([c0, c1, c2]).shuffle(seed=seed)

    # Tokenize split
    tok_train = train_10k.map(tokenize_fn, batched=True)

    # Init fresh model
    print("Init model...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    model.config.pad_token_id = model.config.eos_token_id

    # Train arguments
    args = TrainingArguments(
        output_dir=f"./res_seed_{seed}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=1,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        compute_metrics=compute_metrics,
    )

    # Train
    print("Training...")
    trainer.train()

    # Evaluate on OOD
    print("Evaluating OOD...")
    split_res = {"Split_Seed": seed}
    for name, ds in tok_ood.items():
        preds = trainer.predict(ds)
        acc = preds.metrics['test_accuracy']
        split_res[name] = acc
        print(f"  {name}: {acc:.4f}")

    all_results.append(split_res)

# 5. Results Table & Visualizations
df_results = pd.DataFrame(all_results)
df_results.set_index("Split_Seed", inplace=True)

# Add Mean/Std column
df_mean = df_results.mean().to_frame(name='Average_Acc').T
df_final = pd.concat([df_results, df_mean])

print("\n" + "="*80)
print("FINAL OOD EVALUATION TABLE")
print("="*80)
print(df_final.to_markdown())
print("="*80 + "\n")

# Plot 1: Split Results Comparison
plt.figure(figsize=(10, 5))
df_results.T.plot(kind='bar', ax=plt.gca(), colormap='viridis')
plt.title('Accuracy by Random Split across OOD Datasets')
plt.ylabel('Accuracy')
plt.xlabel('Dataset')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Split Seed')
plt.tight_layout()
plt.show()

# Plot 2: Average Performance Overview
plt.figure(figsize=(8, 4))
df_mean.T['Average_Acc'].sort_values().plot(kind='barh', color='coral', edgecolor='black')
plt.title('Average Accuracy Across Splits')
plt.xlabel('Mean Accuracy')
plt.xlim(0, 1.0)
for index, value in enumerate(df_mean.T['Average_Acc'].sort_values()):
    plt.text(value, index, f" {value:.3f}", va='center')
plt.tight_layout()
plt.show()

print("Execution complete.")

Loading base SNLI train data...


NameError: name 'load_dataset' is not defined

In [3]:
df_final.to_csv("ood_evaluation_results.csv")